# Bölüm 6 — BÖLÜM 6: Sınıflandırma: Karar Ağaçlarından Topluluk Öğrenmesine

**VERİ MADENCİLİĞİ VE MAKİNE ÖĞRENMESİ**  
*Python ile Temel Analitikten Büyük Veri ve Gerçek Zamanlı Sistemlere*

Bu defter, kitabın 6. bölümündeki tüm kod örneklerini içerir. Her hücrenin başlığı kitaptaki alt bölüme karşılık gelir.


In [ ]:
# Bu bölüm için gerekli paketler
!pip install -q catboost lightgbm matplotlib numpy pandas scikit-learn shap xgboost


## 6.1. Temel Sınıflandırıcılar


### Python Uygulaması — Lojistik Regresyon

`bolum06/06_01_01_python-uygulamasi-lojistik-regresyon.py`

_Kitap: Kod 6.1_


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score,
                             RocCurveDisplay)

# ─── 1. Veri Yükleme ──────────────────────────────────────────────
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')  # 1: Malignant, 0: Benign

# ─── 2. Eğitim / Test Bölmesi ────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)  # stratify=y: sınıf dağılımını korur

# ─── 3. Özellik Ölçeklendirme (ZORUNLU) ──────────────────────────
# Gradyan tabanlı optimizasyon için tüm özellikler aynı ölçekte olmalı
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)  # fit + transform
X_test_s  = scaler.transform(X_test)        # sadece transform (veri sızıntısı önlenir)

# ─── 4. Model Eğitimi ─────────────────────────────────────────────
# penalty='l2' (Ridge): varsayılan; C=1.0
# solver='lbfgs': küçük-orta boyutlu veri için optimize edilmiş
# max_iter: konverjans için yeterli iterasyon
lr = LogisticRegression(penalty='l2', C=1.0, solver='lbfgs',
                        max_iter=1000, random_state=42)
lr.fit(X_train_s, y_train)

# ─── 5. Tahmin ve Değerlendirme ───────────────────────────────────
y_pred      = lr.predict(X_test_s)
y_pred_prob = lr.predict_proba(X_test_s)[:, 1]  # P(y=1)

print(f'Doğruluk (Accuracy): {accuracy_score(y_test, y_pred):.4f}')
print(f'ROC-AUC:             {roc_auc_score(y_test, y_pred_prob):.4f}')
print()
print('=== Sınıflandırma Raporu ===')
print(classification_report(y_test, y_pred,
                             target_names=['Benign', 'Malignant']))

# ─── 6. Karışıklık Matrisi ────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
print('Karışıklık Matrisi:')
print(pd.DataFrame(cm, index=['Gerçek 0','Gerçek 1'],
                       columns=['Tahmin 0','Tahmin 1']))

# ─── 7. Katsayı Yorumlama ─────────────────────────────────────────
coef_df = pd.DataFrame({
    'Özellik'   : data.feature_names,
    'Katsayı'   : lr.coef_[0],
    'Odds Oranı': np.exp(lr.coef_[0])
}).sort_values('Katsayı', ascending=False)

print('\nEn etkili 5 pozitif özellik:')
print(coef_df.head())

# ─── 8. Hiperparametre Optimizasyonu: GridSearch ──────────────────
param_grid = {
    'C'      : [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver' : ['liblinear']  # L1 desteği için liblinear kullanılır
}
gs = GridSearchCV(LogisticRegression(max_iter=1000, random_state=42),
                  param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
gs.fit(X_train_s, y_train)
print(f'\nEn iyi parametreler: {gs.best_params_}')
print(f'En iyi CV ROC-AUC: {gs.best_score_:.4f}')


### Python Uygulaması — KNN

`bolum06/06_01_02_python-uygulamasi-knn.py`

_Kitap: Kod 6.2_


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import (train_test_split,
                                      cross_val_score, GridSearchCV)
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# ─── 1. Veri ──────────────────────────────────────────────────────
iris = load_iris()
X = iris.data      # 4 özellik: sepal/petal uzunluk-genişlik
y = iris.target    # 3 sınıf: 0,1,2

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# ─── 2. Ölçeklendirme (ZORUNLU!) ─────────────────────────────────
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# ─── 3. K Değeri Seçimi (Çapraz Doğrulama ile) ───────────────────
k_values = range(1, 31)
cv_scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k, weights='distance')
    scores = cross_val_score(knn, X_train_s, y_train,
                             cv=5, scoring='accuracy')
    cv_scores.append(scores.mean())

best_k = k_values[np.argmax(cv_scores)]
print(f'En iyi K = {best_k}, CV Doğruluk = {max(cv_scores):.4f}')

# ─── 4. Farklı Uzaklık Metriklerini Karşılaştırma ────────────────
metrics = {'euclidean': 2, 'manhattan': 1, 'chebyshev': None}
results = {}

for name, p_val in metrics.items():
    if p_val is not None:
        knn = KNeighborsClassifier(n_neighbors=best_k,
                                   metric='minkowski', p=p_val)
    else:
        knn = KNeighborsClassifier(n_neighbors=best_k,
                                   metric='chebyshev')
    knn.fit(X_train_s, y_train)
    results[name] = knn.score(X_test_s, y_test)

for metric, acc in results.items():
    print(f'{metric:15s}: {acc:.4f}')

# ─── 5. En iyi model ile değerlendirme ───────────────────────────
best_knn = KNeighborsClassifier(n_neighbors=best_k, weights='distance')
best_knn.fit(X_train_s, y_train)
y_pred = best_knn.predict(X_test_s)

print('\n=== Sınıflandırma Raporu ===')
print(classification_report(y_test, y_pred,
                             target_names=iris.target_names))


### Python Uygulaması — Naive Bayes

`bolum06/06_01_03_python-uygulamasi-naive-bayes.py`

_Kitap: Kod 6.3_


In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score)

# Örnek e-posta veri seti
emails = [
    'Bedava kredi kartı teklifi hemen tıkla kazanma fırsatı',
    'Toplantı yarın saat 10da konferans odasında',
    'Büyük indirim sadece bugün yüzde yetmiş ucuz fiyat',
    'Proje raporu ekte gönderiyorum değerlendirmeni bekliyorum',
    'Ücretsiz üyelik hemen kayıt ol ödül kazan',
    'Müşteri şikayetleri için raporun hazır lütfen incele',
    'Para ödülü hemen al kredi başvurusu yap',
    'Akşam yemeği için rezervasyon yaptım saat sekizde',
    'Fatura vadesi geçti gecikme faizi işleniyor hemen öde',
    'Haftalık ekip toplantısı gündemine bakıldı',
]
labels = [1, 0, 1, 0, 1, 0, 1, 0, 1, 0]  # 1: Spam, 0: Ham

# Pipeline: TF-IDF + MultinomialNB
# TF-IDF: Kelime önemini frekansa ve belge sayısına göre ağırlıklandırır
spam_pipeline = Pipeline([
    ('tfidf',       TfidfVectorizer(ngram_range=(1,2),  # Unigram+Bigram
                                   min_df=1,
                                   sublinear_tf=True)),
    ('classifier',  MultinomialNB(alpha=1.0))  # Laplace düzeltmesi
])

# Çapraz doğrulama (küçük veri seti için tüm veri kullanılıyor)
cv_scores = cross_val_score(spam_pipeline, emails, labels,
                             cv=3, scoring='accuracy')
print(f'Çapraz Doğrulama Doğruluk: {cv_scores.mean():.3f}')

spam_pipeline.fit(emails, labels)

# Yeni e-posta tahminleri
yeni_emailler = [
    'Bedava hediye çekilişi kazandınız hemen alın',
    'Yarınki sunum için slaytları paylaşıyorum',
]
tahminler = spam_pipeline.predict(yeni_emailler)
for email, tahmin in zip(yeni_emailler, tahminler):
    etiket = '🚫 SPAM' if tahmin == 1 else '✅ NORMAL'
    print(f'{etiket}: {email[:50]}')

# ─── BÖLÜM B: Sayısal Veri — Gaussian Naive Bayes ────────────────
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler

wine = load_wine()
X_w, y_w = wine.data, wine.target

X_tr, X_te, y_tr, y_te = train_test_split(
    X_w, y_w, test_size=0.2, random_state=42, stratify=y_w
)

# GaussianNB için ölçeklendirme genellikle çok etkili değildir
# (normal dağılım varsayımı mevcutsa), ama yapılması zararlı değildir
gnb = GaussianNB(var_smoothing=1e-9)  # Sayısal kararlılık
gnb.fit(X_tr, y_tr)
y_pred_gnb = gnb.predict(X_te)

print('\n=== Gaussian Naive Bayes — Wine Veri Seti ===')
print(f'Test Doğruluğu: {accuracy_score(y_te, y_pred_gnb):.4f}')
print(classification_report(y_te, y_pred_gnb,
                             target_names=wine.target_names))

# Prior Olasılıklar (modelden okunabilir)
print('Sınıf Prior Olasılıkları:')
for cls, prior in enumerate(gnb.class_prior_):
    print(f'  Sınıf {cls}: {prior:.3f}')


## 6.2. Karar Ağaçları ve Topluluk (Ensemble) Devrimi


### Ağaç Büyütme: Özyinelemeli Bölünme Algoritması

`bolum06/06_02_01_agac-buyutme-ozyinelemeli-bolunme-algoritmasi.txt`

_Kitap: Kod 6.4_


In [ ]:
def build_tree(D, depth=0):
    # Durma koşulları kontrol edilir
    if depth == max_depth: return leaf(majority_class(D))
    if len(D) < min_samples_split: return leaf(majority_class(D))
    if impurity(D) == 0: return leaf(D[0].class)   # Tamamen saf düğüm

# --- Algoritma: Özyinelemeli Karar Ağacı Büyütme ---
    # En iyi bölünme aranır
    best_feat, best_thresh = find_best_split(D)   # max IG veya min Gini

# --- Algoritma: Özyinelemeli Karar Ağacı Büyütme ---
    # Veri bölünür ve alt ağaçlar oluşturulur
    D_left  = D[D[best_feat] <= best_thresh]
    D_right = D[D[best_feat] >  best_thresh]

# --- Algoritma: Özyinelemeli Karar Ağacı Büyütme ---
    node.left  = build_tree(D_left,  depth + 1)
    node.right = build_tree(D_right, depth + 1)
    return node


### Python Uygulaması — Karar Ağacı

`bolum06/06_02_01_python-uygulamasi-karar-agaci.py`

_Kitap: Kod 6.5_


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.metrics import classification_report, accuracy_score

# --- Python: Kapsamlı Karar Ağacı Uygulaması ---
# ─── 1. Veri Hazırlama ────────────────────────────────────────────
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# --- Python: Kapsamlı Karar Ağacı Uygulaması ---
# ─── 2. Temel Model (Budanmamış) ─────────────────────────────────
dt_full = DecisionTreeClassifier(criterion="entropy", random_state=42)
dt_full.fit(X_train, y_train)
print(f"Tam ağaç - Eğitim: {dt_full.score(X_train, y_train):.4f}, Test: {dt_full.score(X_test, y_test):.4f}")

# --- Python: Kapsamlı Karar Ağacı Uygulaması ---
# ─── 3. Ön-Budama (max_depth) ────────────────────────────────────
dt_pruned = DecisionTreeClassifier(
    criterion="entropy", max_depth=4,
    min_samples_split=10, min_samples_leaf=5,
    random_state=42)
dt_pruned.fit(X_train, y_train)
print(f"Budanmış ağaç - Test: {dt_pruned.score(X_test, y_test):.4f}")

# --- Python: Kapsamlı Karar Ağacı Uygulaması ---
# ─── 4. Sonradan-Budama: Cost Complexity Pruning ─────────────────
path = dt_full.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas[:-1]   # Son değer (tam budama) hariç

# --- Python: Kapsamlı Karar Ağacı Uygulaması ---
cv_scores = []
for alpha in ccp_alphas:
    dt = DecisionTreeClassifier(ccp_alpha=alpha, random_state=42)
    scores = cross_val_score(dt, X_train, y_train, cv=5)
    cv_scores.append(scores.mean())

# --- Python: Kapsamlı Karar Ağacı Uygulaması ---
best_alpha = ccp_alphas[np.argmax(cv_scores)]
print(f"En iyi ccp_alpha: {best_alpha:.6f}")

# --- Python: Kapsamlı Karar Ağacı Uygulaması ---
dt_ccp = DecisionTreeClassifier(ccp_alpha=best_alpha, random_state=42)
dt_ccp.fit(X_train, y_train)
print(f"CCP Budanmış - Test: {dt_ccp.score(X_test, y_test):.4f}")

# --- Python: Kapsamlı Karar Ağacı Uygulaması ---
# ─── 5. Özellik Önemi Analizi ────────────────────────────────────
feat_imp = pd.Series(dt_ccp.feature_importances_,
                     index=data.feature_names).sort_values(ascending=False)
print("\nEn önemli 5 özellik:")
print(feat_imp.head())

# --- Python: Kapsamlı Karar Ağacı Uygulaması ---
# ─── 6. Karar Kurallarını Metin Olarak Görüntüleme ───────────────
print("\nKarar Ağacı Kuralları (ilk 3 seviye):")
print(export_text(dt_ccp, feature_names=list(data.feature_names),
                  max_depth=3))

# --- Python: Kapsamlı Karar Ağacı Uygulaması ---
# ─── 7. Sınıflandırma Raporu ─────────────────────────────────────
y_pred = dt_ccp.predict(X_test)
print(classification_report(y_test, y_pred,
      target_names=["Benign", "Malignant"]))


### Python Uygulaması — Random Forest ve OOB

`bolum06/06_02_02_python-uygulamasi-random-forest-ve-oob.py`

_Kitap: Kod 6.6_


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.inspection import permutation_importance

# --- Python: Random Forest — Kapsamlı Uygulama ---
# ─── 1. Veri ──────────────────────────────────────────────────────
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# --- Python: Random Forest — Kapsamlı Uygulama ---
# ─── 2. Temel Random Forest + OOB Skoru ──────────────────────────
rf = RandomForestClassifier(
    n_estimators=200,
    max_features="sqrt",
    oob_score=True,          # OOB genelleme tahmini
    n_jobs=-1,               # Tüm CPU çekirdeklerini kullan
    random_state=42)
rf.fit(X_train, y_train)

# --- Python: Random Forest — Kapsamlı Uygulama ---
print(f"OOB Skoru:  {rf.oob_score_:.4f}")
print(f"Test Skoru: {rf.score(X_test, y_test):.4f}")

# --- Python: Random Forest — Kapsamlı Uygulama ---
# ─── 3. n_estimators Artışının Etkisi (Hata Analizi) ─────────────
oob_errors = []
for n in range(10, 301, 10):
    rf_n = RandomForestClassifier(n_estimators=n, oob_score=True,
                                   n_jobs=-1, random_state=42)
    rf_n.fit(X_train, y_train)
    oob_errors.append(1 - rf_n.oob_score_)
# n artıkça OOB hatası düşer, yaklaşık n=100-150 civarında plato oluşur

# --- Python: Random Forest — Kapsamlı Uygulama ---
# ─── 4. RandomizedSearchCV ile Hiperparametre Optimizasyonu ───────
param_dist = {
    "n_estimators"    : [100, 200, 300],
    "max_features"    : ["sqrt", "log2", 0.3, 0.5],
    "max_depth"       : [None, 10, 20, 30],
    "min_samples_leaf": [1, 2, 5, 10],
    "class_weight"    : [None, "balanced"]
}
rscv = RandomizedSearchCV(
    RandomForestClassifier(oob_score=True, n_jobs=-1, random_state=42),
    param_distributions=param_dist,
    n_iter=30, cv=5, scoring="roc_auc",
    n_jobs=-1, random_state=42)
rscv.fit(X_train, y_train)
print(f"En iyi parametreler: {rscv.best_params_}")
print(f"En iyi CV ROC-AUC:  {rscv.best_score_:.4f}")

# --- Python: Random Forest — Kapsamlı Uygulama ---
# ─── 5. Özellik Önemi (Gini + Permütasyon) ───────────────────────
best_rf = rscv.best_estimator_

# --- Python: Random Forest — Kapsamlı Uygulama ---
# Gini tabanlı özellik önemi
gini_imp = pd.Series(best_rf.feature_importances_,
                     index=data.feature_names).sort_values(ascending=False)
print("\nGini Tabanlı En Önemli 5 Özellik:")
print(gini_imp.head())

# --- Python: Random Forest — Kapsamlı Uygulama ---
# Permütasyon tabanlı özellik önemi (daha güvenilir)
perm_imp = permutation_importance(
    best_rf, X_test, y_test, n_repeats=15,
    random_state=42, scoring="roc_auc")
perm_df = pd.DataFrame({
    "importance_mean": perm_imp.importances_mean,
    "importance_std" : perm_imp.importances_std
}, index=data.feature_names).sort_values("importance_mean", ascending=False)
print("\nPermütasyon Tabanlı En Önemli 5 Özellik:")
print(perm_df.head())

# --- Python: Random Forest — Kapsamlı Uygulama ---
# ─── 6. Nihai Değerlendirme ───────────────────────────────────────
y_pred = best_rf.predict(X_test)
y_prob = best_rf.predict_proba(X_test)[:,1]
print(f"\nTest Doğruluğu: {roc_auc_score(y_test, y_prob):.4f} (ROC-AUC)")
print(classification_report(y_test, y_pred,
      target_names=["Benign","Malignant"]))


### Python Uygulaması — Boosting Algoritmaları

`bolum06/06_02_03_python-uygulamasi-boosting-algoritmalari.py`

_Kitap: Kod 6.7_


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import (AdaBoostClassifier,
                               GradientBoostingClassifier)
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, classification_report

# --- Python: AdaBoost ve Gradient Boosting ---
# ─── Ortak Veri Hazırlama ─────────────────────────────────────────
data = load_breast_cancer()
X, y = pd.DataFrame(data.data, columns=data.feature_names), data.target
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

# --- Python: AdaBoost ve Gradient Boosting ---
# ─── AdaBoost ─────────────────────────────────────────────────────
# base_estimator: max_depth=1 (stump) varsayılan zayıf öğrenici
ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=200,
    learning_rate=0.5,
    random_state=42)
ada.fit(X_tr, y_tr)
print(f"AdaBoost ROC-AUC: {roc_auc_score(y_te, ada.predict_proba(X_te)[:,1]):.4f}")

# --- Python: AdaBoost ve Gradient Boosting ---
# Staging: Her iterasyonda test hatasını takip et
staged_auc = [roc_auc_score(y_te, pred[:,1])
              for pred in ada.staged_predict_proba(X_te)]
best_n = np.argmax(staged_auc) + 1
print(f"En iyi n_estimators: {best_n}, AUC: {max(staged_auc):.4f}")

# --- Python: AdaBoost ve Gradient Boosting ---
# ─── Sklearn GradientBoosting ────────────────────────────────────
gbm = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,          # Stochastic GB: alt-örnekleme ile hız+çeşitlilik
    max_features="sqrt",    # Her bölünmede rastgele özellik alt kümesi
    random_state=42)
gbm.fit(X_tr, y_tr)
print(f"GBM  ROC-AUC: {roc_auc_score(y_te, gbm.predict_proba(X_te)[:,1]):.4f}")

# --- Python: XGBoost — Erken Durdurma ve SHAP ---
import xgboost as xgb
# pip install xgboost

# --- Python: XGBoost — Erken Durdurma ve SHAP ---
# ─── XGBoost ─────────────────────────────────────────────────────
xgb_clf = xgb.XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,   # Her ağaçta kullanılacak özellik oranı
    reg_alpha=0.1,          # L1 düzenlileştirme
    reg_lambda=1.0,         # L2 düzenlileştirme
    use_label_encoder=False,
    eval_metric="auc",
    random_state=42)

# --- Python: XGBoost — Erken Durdurma ve SHAP ---
# Erken Durdurma (Early Stopping): Doğrulama seti ile fazla ağacı engelle
xgb_clf.fit(X_tr, y_tr,
            eval_set=[(X_te, y_te)],
            early_stopping_rounds=20,   # 20 ardışık gerileme toleransı
            verbose=False)

# --- Python: XGBoost — Erken Durdurma ve SHAP ---
print(f"XGBoost En iyi iterasyon: {xgb_clf.best_iteration}")
print(f"XGBoost ROC-AUC: {roc_auc_score(y_te, xgb_clf.predict_proba(X_te)[:,1]):.4f}")

# --- Python: XGBoost — Erken Durdurma ve SHAP ---
# ─── SHAP Değerleri ile Yorumlanabilirlik ────────────────────────
# pip install shap
import shap

# --- Python: XGBoost — Erken Durdurma ve SHAP ---
explainer = shap.TreeExplainer(xgb_clf)
shap_values = explainer.shap_values(X_te)

# --- Python: XGBoost — Erken Durdurma ve SHAP ---
# Global özellik önemi (SHAP bazlı)
shap_imp = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=data.feature_names).sort_values(ascending=False)
print("\nSHAP Tabanlı En Önemli 5 Özellik:")
print(shap_imp.head())

# --- Python: XGBoost — Erken Durdurma ve SHAP ---
# Tek bir örnek için yerel yorum
print("\nÖrnek 0 için SHAP açıklaması (ilk 5 özellik):")
for feat, sv in zip(data.feature_names[:5], shap_values[0][:5]):
    print(f"  {feat:35s}: {sv:+.4f}")

# --- Python: LightGBM ve CatBoost ---
import lightgbm as lgb
# pip install lightgbm catboost
from catboost import CatBoostClassifier

# --- Python: LightGBM ve CatBoost ---
# ─── LightGBM ────────────────────────────────────────────────────
lgbm_clf = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,          # Yaprak sayısı; max_depth yerine ana kontrol
    max_depth=-1,           # -1 = sınırsız (num_leaves ile kontrol)
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    verbose=-1)

# --- Python: LightGBM ve CatBoost ---
lgbm_clf.fit(X_tr, y_tr,
             eval_set=[(X_te, y_te)],
             callbacks=[lgb.early_stopping(20), lgb.log_evaluation(0)])

# --- Python: LightGBM ve CatBoost ---
print(f"LightGBM ROC-AUC: {roc_auc_score(y_te, lgbm_clf.predict_proba(X_te)[:,1]):.4f}")

# --- Python: LightGBM ve CatBoost ---
# ─── CatBoost ────────────────────────────────────────────────────
# Kategorik değişkenleri belirtmek yeterli; otomatik kodlar
cat_clf = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3.0,
    random_seed=42,
    verbose=0)

# --- Python: LightGBM ve CatBoost ---
cat_clf.fit(X_tr, y_tr,
            eval_set=(X_te, y_te),
            early_stopping_rounds=20)

# --- Python: LightGBM ve CatBoost ---
print(f"CatBoost  ROC-AUC: {roc_auc_score(y_te, cat_clf.predict_proba(X_te)[:,1]):.4f}")

# --- Python: LightGBM ve CatBoost ---
# ─── Tüm Modelleri Karşılaştır ───────────────────────────────────
models = {
    "AdaBoost"       : ada,
    "GBM"            : gbm,
    "XGBoost"        : xgb_clf,
    "LightGBM"       : lgbm_clf,
    "CatBoost"       : cat_clf,
}
print("\n=== Model Karşılaştırması ===")
print(f"{'Model':15s}  {'ROC-AUC':>8s}  {'Accuracy':>9s}")
print("-" * 38)
for name, model in models.items():
    proba = model.predict_proba(X_te)[:,1]
    pred  = model.predict(X_te)
    auc   = roc_auc_score(y_te, proba)
    acc   = (pred == y_te).mean()
    print(f"{name:15s}  {auc:>8.4f}  {acc:>9.4f}")
